In [1]:
using Pkg
using Metal

cd(@__DIR__)
Pkg.activate("../")
ParamFile = "../config/testparam.csv"  # maybe GeoPoints and planet1D should be fusioned

# batchGPU should be at this level (I have not made it as a module yet, since the choice of Metal/CUDA should be done in a manual way)
include("../src/batchFiles/batchGPU.jl")


include("../src/commonBatchs.jl")
#include("../src/flexOPT.jl")
include("../src/planet1D.jl")
include("../src/GeoPoints.jl")

  Activating 

devs = Metal.devices() = Metal.MTL.MTLDeviceInstance[Metal.MTL.MTLDeviceInstance (object of type AGXG13XDevice)]

project at `~/Documents/Github/flexOPT`



→ Using Metal backend (1 device(s))
Selected backend type: MetalBackend


Main.GeoPoints

In [2]:
using .commonBatchs, UnPack, Symbolics

In [3]:
include("../src/motorsOPT/BsplineHelpers.jl")

evaluate_bspline_piecewise_deriv (generic function with 1 method)

In [ ]:
orderBspline1D = 2
YorderBspline1D = 2
#the coordinate in Δy is (as usual) with the offset 1
μᶜs = [1,1.5,2.5,3] # ∈ R for variable, the nodes for interpolated Taylor expansions
μs = [1,1.5,2.5,3] # ∈ R for field, the nodes for interpolated Taylor expansions
nodesFromOne = [1,2,3] # ∈ Z like [1,2,3], an array of integers collect(1:1:N) (nothing else!!)
ν = [2]

1-element Vector{Int64}:
 2

In [ ]:


allNodes = unique(sort(vcat(
    Float64.(nodesFromOne),
    Float64.(ν),
    Float64.(μs),
    Float64.(μᶜs),
)))

to_indices(xs, master) = searchsortedfirst.(Ref(master), Float64.(xs))

idx_nodesFromOne = to_indices(nodesFromOne, allNodes)
idx_ν            = to_indices(ν, allNodes)
idx_μs           = to_indices(μs, allNodes)
idx_μᶜs          = to_indices(μᶜs, allNodes)


params  = (maximumOrder=orderBspline1D, allNodes = allNodes, idx_nodesNum = idx_nodesFromOne, idx_refPoints = idx_μs, idx_selectedPoints = idx_μs)


# idx_nodesNum : an ordinary consecutive integer increment from 1 (the numerical nodes with Δy)
# idx_refPoints_original : supporting nodes to construct the Bspline family
# idx_selectedPoints  : the node addresses that user needs to take, should be a subset of idx_refPoints_original

In [ ]:

Δ = 1.0 # Float64
l_n_max = 4 # variable
lᶜ_nᶜ_max = 4 # field

paramsforWYYKKIntegral = (orderBspline1D=orderBspline1D, YorderBspline1D=YorderBspline1D, μᶜs=μᶜs, μs=μs, nodesFromOne=nodesFromOne ν=ν, Δ=Δ, l_n_max=l_n_max, lᶜ_nᶜ_max=lᶜ_nᶜ_max)

In [ ]:

function WYYKKIntegralIn1D(paramsforWYYKKIntegral)
    
    # Δ should be strictly Float64

    # orders: -1 -> indicator function, 0 -> box car, >=1 -> B-spline

    # this computes the analytical value of the 1D integral between B-spline fns and weighted Taylor kernels
    # \int dx Bspline Y_μᶜ Y_μ  K_{lᶜ-nᶜ}(y-y_μᶜ) K_{l-n}(y-y_μ)

    # unlike the previous integralBsplineTaylorKernels1D, it computes for a specific ν
    # Cˡη;μ are computed for a specific geometry, so even though the boundary condition reduces
    # the number of available points, each Taylor expansion for K_{l-n}(y-y_μ) should be Ok

    @unpack orderBspline1D, YorderBspline1D, μᶜs, μs, nodesFromOne, ν, Δ, l_n_max, lᶜ_nᶜ_max = paramsforWYYKKIntegral
    
    allNodes  = vcat(Float64.(nodesFromOne), Float64.(ν), μs, μᶜs )
    allNodes = unique(sort(allNodes))
    

    # if numberNodes = -1 then we are thinking about an infinitely large segment (for W it should be the case except boundaries)

    for μᶜ ∈ μᶜs, μ ∈ μ, lᶜ_nᶜ ∈ lᶜ_nᶜ_list, l_n ∈ l_n_list
        paramsν  = (maximumOrder=orderBspline1D, nodesNum = nodesFromOne, allNodes = allNodes, refPoints = ν)
        paramsμᶜ = (maximumOrder=YorderBspline1D, nodesNum = nodesFromOne, allNodes = allNodes, refPoints = μᶜs)
        paramsμ  = (maximumOrder=YorderBspline1D, nodesNum = nodesFromOne, allNodes = allNodes, refPoints = μs)

    end



    maximumOrder = maximum((orderBspline1D,YorderBspline1D,0)) # even if we only use the indicator functions (classical FD)
    params=@strdict maximumOrder numberNodes 

    #output,_=@produce_or_load(BsplineTimesPolynomialsIntegrated,params,datadir("BsplineInt");filename = config -> savename("Bspline",params))
    
    #output=myProduceOrLoad(BsplineTimesPolynomialsIntegrated,params,"BsplineInt","Bspline")
    @show output=BsplineTimesPolynomialsIntegrated(params)

    nodeIndices,nodesSymbolic,b_deriv,integral_b,Δx,extFns,x,modμ =output["BsplineIntegraters"]

    @show "coucou "
    @show modμ
    if orderBspline1D=== -1
        # this is for an indicator function
        if l_n_variable === 0 && l_n_field === 0
            kernelValue=1.0
        else
            kernelValue=0.0
        end

    else
        
        
        # here we make a function Y_μ' Y_μ K_μ' K_μ (details ommitted)
        # note that ν is somewhere middle or at extremeties and 'ν+' expression is ommitted 

        Y_μᶜ = zeros(Num,L)
        Y_μ = zeros(Num,L)

        if YorderBspline1D === -1

            if μᶜ === ν
                Y_μᶜ = ones(Num,L) 
            end

            if μ === ν
                Y_μ = ones(Num,L)
            end

        else
            Y_μᶜ=b_deriv[:,μᶜ,1,YorderBspline1D+1]
            Y_μ =b_deriv[:,μ ,1,YorderBspline1D+1]
        end

        # modμ[3,:,ι+1] is the symbolic expression of the centre to compute Taylor kernels, which can be staggered!!!

        K_μᶜ=(x-Δx*modμ[3,μᶜ,YorderBspline1D+1])^l_n_variable
        K_μ =(x-Δx*modμ[3,μ,YorderBspline1D+1])^l_n_field

        # the convoluted function of all above
        F = mySimplify.(Y_μᶜ .* Y_μ .* K_μᶜ .* K_μ)


        # the target kernel integral

        targetKernel = integral_b[ν,orderBspline1D+1]

        dictionaryForSubstitute = Dict()
    
        for i in 0:1:orderBspline1D
            F = integrateTaylorPolynomials.(F,x) # integrate already for the 1st partial of W
            
            # mathematically I need to understand why but F cannot be disturbed by supplementary complexities due to constants 
            # that are arbitrarily put during the integral
            
            #F .-= substitute(F[ν],Dict(x=>nodesSymbolic[ν]))

            # F should be continuous, in general but since all the integral by parts
            # is done piecewisely we might not need this
            #=
            for iSegment in 2:1:L # this should be sequential
                lastValue = substitute(F[iSegment-1],Dict(x=>nodesSymbolic[iSegment]))
                startValue = substitute(F[iSegment],Dict(x=>nodesSymbolic[iSegment]))
                shiftValue = lastValue - startValue
                F[iSegment]=F[iSegment]+shiftValue
            end
            =#

            F .= mySimplify(F)

            for iSegment in nodeIndices
                dictionaryForSubstitute[extFns[1,iSegment,i+1]]=substitute(F[iSegment],Dict(x=>nodesSymbolic[iSegment]))
                dictionaryForSubstitute[extFns[2,iSegment,i+1]]=substitute(F[iSegment],Dict(x=>nodesSymbolic[iSegment+1]))
            end
        end

        #@show dictionaryForSubstitute,targetKernel
        

        kernelValue = substitute(targetKernel,dictionaryForSubstitute)  
        
        kernelValue = substitute(kernelValue,Dict(Δx=>Δ))/(BigInt(factorial(l_n_field))*BigInt(factorial(l_n_variable)))
        kernelValue = Num2Float64(kernelValue)
        
        #a= (Δ^(l_n_variable+l_n_field+1)-(-Δ)^(l_n_variable+l_n_field+1))/((l_n_variable+l_n_field+2)*(l_n_variable+l_n_field+1)*factorial(BigInt(l_n_variable))*factorial(BigInt(l_n_field)))
        #@show a
    end
    return kernelValue,modμ
    
end

In [ ]:

result = constructBsplineFamily(params)

fig = plot_bspline_family(result; order=0, N=50,show_full=false)
display(fig)

b = result.b

In [ ]:
plot_bspline_family(result; order=2, derivOrder=1, N=50)

In [ ]:
plot_bspline_family(result; order=5, derivOrder=3, N=50)

In [ ]:
orderBspline1D = 1
YorderBspline1D = 1
#the coordinate in Δy is (as usual) with the offset 1
μᶜs = [2.0] # ∈ R for variable
μs = [1.5,2.5] # ∈ R for field
nodesFromOne = [1,2,3] # ∈ Z like [1,2,3]
ν = [2]



allNodes = unique(sort(vcat(
    Float64.(nodesFromOne),
    Float64.(ν),
    Float64.(μs),
    Float64.(μᶜs),
)))

to_indices(xs, master) = searchsortedfirst.(Ref(master), Float64.(xs))

idx_nodesFromOne = to_indices(nodesFromOne, allNodes)
idx_ν            = to_indices(ν, allNodes)
idx_μs           = to_indices(μs, allNodes)
idx_μᶜs          = to_indices(μᶜs, allNodes)

paramsν  = (maximumOrder=orderBspline1D, allNodes = allNodes, idx_nodesNum = idx_nodesFromOne, idx_refPoints = idx_ν)

params = paramsν

#######
# inside constructBsplineFamily

@unpack maximumOrder, allNodes, idx_nodesNum, idx_refPoints = params
@variables x Δx ξ
   
# setting
NorderShiftedByTwo = maximumOrder+2 # indicator function for -1 and 0-th B-spline for 0
numberNodes = length(allNodes)
numberFunctionsOriginal = length(idx_refPoints)
idx_refPointsOriginal = idx_refPoints

leftPlusNumberFunctions = 0 # we put the 'real' leftmost point in the numerical node system 
rightPlusNumberFunctions = 0 # idem

allNodesSymbolic = Δx .* allNodes 

extFns = Symbolics.variables(:extFns,1:2,1:numberNodes,1:NorderShiftedByTwo)
∂x = Differential(x)

# examination of leftmost and rightmost points (the only point where we need idx_nodesNum)

if allNodes[idx_refPoints[1]]>allNodes[idx_nodesNum[1]]
    idx_refPoints = vcat(idx_nodesNum[1],idx_refPoints)
    leftPlusNumberFunctions = 1
end

if allNodes[idx_refPoints[end]]<allNodes[idx_nodesNum[end]]
    idx_refPoints = vcat(idx_refPoints, idx_nodesNum[end])
    rightPlusNumberFunctions = 1
end

numberFunctions = numberFunctionsOriginal + leftPlusNumberFunctions + rightPlusNumberFunctions

indicesForEachFunction = Array{Integer,2}(undef, 2, numberFunctions)


physicalKnots = allNodes[idx_refPoints]
extendedKnots = vcat(
    fill(physicalKnots[1], maximumOrder),
    physicalKnots,
    fill(physicalKnots[end], maximumOrder),
)

extendedKnotsSymbolic = Δx .* extendedKnots

denominators = Array{Num,3}(undef, 2, numberFunctions, maximumOrder)

for idx in 1:numberFunctions
    for p in 1:maximumOrder
        denominators[1,idx,p] = extendedKnotsSymbolic[idx+p]   - extendedKnotsSymbolic[idx]
        denominators[2,idx,p] = extendedKnotsSymbolic[idx+p+1] - extendedKnotsSymbolic[idx+1]
    end
end



for i in 1:numberFunctions-1
    startIdx=idx_refPoints[i]
    endIdx=idx_refPoints[i+1]
    indicesForEachFunction[1,i]=startIdx
    indicesForEachFunction[2,i]=endIdx
    for j in 1:maximumOrder
       denominators[1] = # upgoing
       denominators[2] = # downgoing
    end
end
indicesForEachFunction[1,numberFunctions] = idx_refPoints[numberFunctions]
indicesForEachFunction[2,numberFunctions] = idx_refPoints[numberFunctions]
deltaX[i] = Δx # dummy
# b-splines

b = zeros(Num, numberNodes, numberFunctions, NorderShiftedByTwo)
bX = zeros(Num,numberNodes, 2, NorderShiftedByTwo)

# b-splines derivatives

b_deriv = zeros(Num,numberNodes,numberFunctions,NorderShiftedByTwo,NorderShiftedByTwo)


# indicator function 

b[idx_nodesNum[1]:idx_nodesNum[end],:,1] .= 1.0

# B-spline functions

for ι in 0:1:NorderShiftedByTwo-2

    if ι == 0

        for idx in 1:numberFunctions
            startIdx = indicesForEachFunction[1,idx]
            endIdx = indicesForEachFunction[2,idx]
            b[startIdx:endIdx,idx,ι+2] .= 1.0
        end

    else

        for idx in 1:numberFunctions
            up = denominators[1,idx,ι]
            down = denominators[2,idx,ι]

            for νSegment in 1:numberNodes
                val = zero(Num)

                if !iszero(up)
                    num_up = x - extendedKnotsSymbolic[idx]
                    val += mySimplify(num_up / up * b[νSegment, idx, ι+1])
                end

                if idx < numberFunctions && !iszero(down)
                    num_down = extendedKnotsSymbolic[idx+ι+1] - x
                    val += mySimplify(num_down / down * b[νSegment, idx+1, ι+1])
                end

                b[νSegment, idx, ι+2] = val
            end

        end
    end

    # unfortunately the νₗ and νᵣ related functions should recover the 'lost' amplitudes due to the trunctation 
    # we need to re-build based on the residual

    for ν in nodeIndices
        tmpν = ν - νₗ + 1
        
        #b[:,tmpν,ι+2] .= 0
        if ν===νₗ  # this loop runs only for the extremeties!!!!!
            # the nodes involved at most are: νₗ, νₗ + 1, ⋯, νₗ + ceiling 
            bX[1:ceiling+1,1,ι+2] .=1
            for νSegment in 1:1:ceiling+1
                for νFunction in 2:1:numberNodes
                    bX[νSegment,1,ι+2] -= b[νSegment,νFunction,ι+2]
                end
            end
        elseif ν===νᵣ
            bX[numberNodes-ceiling-1:numberNodes,2,ι+2] .=1
            for νSegment in numberNodes-ceiling-1:1:numberNodes
                for νFunction in 1:1:numberNodes-1
                    bX[νSegment,2,ι+2] -= b[νSegment,νFunction,ι+2]
                enim d
            end
        end

    end


    b[:,1,ι+2]=bX[:,1,ι+2]
    b[:,end,ι+2]=bX[:,end,ι+2]
    
        
    
    # computing the derivatives 

    for i in 0:1:maximumOrder-1 
        if i === 0
            b_deriv[:,:,i+1,ι+2] = b[:,:,ι+2]
        else
            b_deriv[:,:,i+1,ι+2] = mySimplify.(∂x.(b_deriv[:,:,i,ι+2]))
        
        end

    end

end
    

In [ ]:


function BsplineTimesPolynomialsIntegrated(params::Dict)

    @unpack maximumOrder, numberNodes = params
    @variables x Δx ξ
   
    extFns = Symbolics.variables(:extFns,1:2,1:numberNodes,1:maximumOrder+1)
    # maximum order of B-spline
    
    maximumOrder = maximumOrder + 1

    ∂x = Differential(x)
    
    # input parameters
    
  
    
    # the left and right indices
    
    νₗ = 1 # we cannot touch this anymore
    νᵣ = numberNodes # like in the middle we know that the expression is ok 

    
    # let's start
    
    
    nodeIndices = collect(νₗ:1:νᵣ)
    nodesSymbolic = Δx .* nodeIndices
    append!(nodesSymbolic,nodesSymbolic[end])


    
    # b-splines
    
    b = zeros(Num, numberNodes, numberNodes, maximumOrder)
    bX = zeros(Num,numberNodes, 2, maximumOrder)
    
    # b-splines derivatives
    
    b_deriv = zeros(Num,numberNodes, numberNodes,maximumOrder,maximumOrder)
   
    for ι in 0:1:maximumOrder-1
        
        local neighbour = Int((1 - (-1)^ι) / 2)
        floor = Int((ι - neighbour) / 2)
        ceiling = Int((ι + neighbour) / 2)
        if ι === 0
            for ν in nodeIndices # this will run for all the ν related to nodes
                tmpν = ν - νₗ + 1
            
                b[tmpν, tmpν, ι+1] = 1
            end
        else
    
            for ν in nodeIndices # this will run for all the ν related to nodes
                tmpν = ν - νₗ + 1
                # the denominator for the ι>0
    
                denominator = BigInt(ι) * Δx
    
                # for the upgoing part
    
                if ν - neighbour <= νᵣ && ν - neighbour >= νₗ
                    rightlimit = minimum((numberNodes, tmpν + floor))
                    leftlimit = maximum((1, tmpν - ceiling))
                    for νSegment in leftlimit:1:rightlimit
                        tmpνSegment = νSegment #- νₗ + 1
                        numerator = x - BigInt(ν - ceiling) * Δx
                        b[tmpνSegment, tmpν, ι+1] += mySimplify(numerator / denominator * b[tmpνSegment, tmpν-neighbour, ι])
                    end
                end
    
                #for the downgoing part
    
                if ν - neighbour + 1 <= νᵣ && ν - neighbour + 1 >= νₗ
                    rightlimit = minimum((numberNodes, tmpν + floor + 1))
                    leftlimit = maximum((1, tmpν - ceiling + 1))
                    for νSegment in leftlimit:1:rightlimit
                        tmpνSegment = νSegment #- νₗ + 1
                        numerator = BigInt(ν + floor + 1) * Δx - x
                        b[tmpνSegment, tmpν, ι+1] += mySimplify(numerator / denominator * b[tmpνSegment, tmpν-neighbour+1, ι])
                    end
    
                end
    
            end
        end
    
        # unfortunately the νₗ and νᵣ related functions should recover the 'lost' amplitudes due to the trunctation 
        # we need to re-build based on the residual
    
        for ν in nodeIndices
            tmpν = ν - νₗ + 1
            
            #b[:,tmpν,ι+1] .= 0
            if ν===νₗ  # this loop runs only for the extremeties!!!!!
                # the nodes involved at most are: νₗ, νₗ + 1, ⋯, νₗ + ceiling 
                bX[1:ceiling+1,1,ι+1] .=1
                for νSegment in 1:1:ceiling+1
                    for νFunction in 2:1:numberNodes
                        bX[νSegment,1,ι+1] -= b[νSegment,νFunction,ι+1]
                    end
                end
            elseif ν===νᵣ
                bX[numberNodes-ceiling-1:numberNodes,2,ι+1] .=1
                for νSegment in numberNodes-ceiling-1:1:numberNodes
                    for νFunction in 1:1:numberNodes-1
                        bX[νSegment,2,ι+1] -= b[νSegment,νFunction,ι+1]
                    end
                end
            end
    
        end
    

        b[:,1,ι+1]=bX[:,1,ι+1]
        b[:,end,ι+1]=bX[:,end,ι+1]
        
          
        
        # computing the derivatives 
    
        for i in 0:1:maximumOrder-1 
            if i === 0
                b_deriv[:,:,i+1,ι+1] = b[:,:,ι+1]
            else
                b_deriv[:,:,i+1,ι+1] = mySimplify.(∂x.(b_deriv[:,:,i,ι+1]))
            
            end
    
        end
    
    end
        
    
    # re-calculated centre
    
    modμ=zeros(Num,3,numberNodes,maximumOrder) # min, max, mid for each b-splines this can be a multiple of Δx / 2 

    for ι in 0:1:maximumOrder-1
        for μ in nodeIndices
            leftLimit = nodesSymbolic[findfirst(x->x!==0,b[:,μ,ι+1])] /Δx
            rightLimit = nodesSymbolic[findlast(x->x!==0,b[:,μ,ι+1])+1] /Δx
            midNode = mySimplify((leftLimit+rightLimit)/2)
            modμ[1,μ,ι+1]=leftLimit 
            modμ[2,μ,ι+1]=rightLimit
            modμ[3,μ,ι+1]=midNode
        end
    end
    

    integral_b= zeros(Num,numberNodes,maximumOrder)

    #gvec = (@variables (g(x))[1:maximumOrder])[1]
    
    

    for ι in 0:1:maximumOrder-1
        for i in 0:1:maximumOrder-1
            for ν in nodeIndices # this will run for all the ν related to nodes
                tmpν = ν - νₗ + 1
                for νSegment in nodeIndices
                    tmpνSegment = νSegment - νₗ + 1
                    
                    integral_b[tmpν,ι+1] -= (-1)^(i)*extFns[1,tmpνSegment,i+1]*substitute(b_deriv[tmpνSegment,tmpν,i+1,ι+1],Dict(x=>nodesSymbolic[tmpνSegment]))
    
                    integral_b[tmpν,ι+1] += (-1)^(i)*extFns[2,tmpνSegment,i+1]*substitute(b_deriv[tmpνSegment,tmpν,i+1,ι+1],Dict(x=>nodesSymbolic[tmpνSegment+1]))
                    
                end
            end
        end
    end
    
    
    integral_b=mySimplify.(integral_b)
    
    BsplineIntegraters=(nodeIndices=nodeIndices,nodesSymbolic=nodesSymbolic,b_deriv=b_deriv,integral_b=integral_b,Δx=Δx,extFns=extFns,x=x,modμ=modμ)
    return @strdict(BsplineIntegraters)

end
    